In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "TCHOL_E.xpt"
csv_file = DATA_DIR / "TCHOL_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve untouched source for fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 TCHOL_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXTC",
    "LBDTCSI"
]


if df.shape != (8132, 3):
    raise ValueError(
        f"Unexpected TCHOL_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "TCHOL_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: TCHOL_E structure = "
    "8,132 rows x 3 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# SEQN:
# identifier -> clean integer representation
#
# LBXTC:
# Total Cholesterol mg/dL
#
# LBDTCSI:
# Total Cholesterol mmol/L
#
# BOTH cholesterol fields are LAB MEASUREMENTS.
#
# DO NOT:
#
# - convert LBXTC to Int64
# - convert LBDTCSI to Int64
# - round()
# - trim legitimate ".0"
# - use float_format
# - recalculate LBDTCSI
#

INTEGER_COLUMNS = [
    "SEQN"
]


MEASUREMENT_COLUMNS = [
    "LBXTC",
    "LBDTCSI"
]


print(
    "\nIdentifier fields:",
    INTEGER_COLUMNS
)

print(
    "Lab measurements preserved AS IS:",
    MEASUREMENT_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT value(s) found. "
        "Do NOT automatically replace them with zero."
    )


print(
    "PASS: TCHOL_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "Unexpected source zero(s) found."
    )


print(
    "PASS: Original TCHOL_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. LBXTC VALID / MISSING CHECK
# ============================================================

#
# CDC HTML:
#
# Valid   = 7,387
# Missing = 745
# Range   = 44-460
#
#
# ATTACHED XPT:
#
# Valid   = 7,387
# Missing = 745
# Range   = 78-460
#
#
# IMPORTANT:
#
# Do NOT change XPT values to force the CDC HTML minimum.
# XPT is the source of truth for fidelity.
#

s = df["LBXTC"]


lbx_valid = int(
    s.notna().sum()
)


lbx_missing = int(
    s.isna().sum()
)


lbx_min = s.min()
lbx_max = s.max()


print(
    "\n--- LBXTC CHECK ---"
)

print(
    "Valid:",
    f"{lbx_valid:,}"
)

print(
    "Missing:",
    f"{lbx_missing:,}"
)

print(
    "Attached XPT minimum:",
    lbx_min
)

print(
    "Attached XPT maximum:",
    lbx_max
)


if lbx_valid != 7387:

    raise ValueError(
        "LBXTC valid count differs."
    )


if lbx_missing != 745:

    raise ValueError(
        "LBXTC missing count differs."
    )


if lbx_min != 78:

    raise ValueError(
        f"Unexpected attached-XPT LBXTC minimum: {lbx_min}"
    )


if lbx_max != 460:

    raise ValueError(
        f"Unexpected LBXTC maximum: {lbx_max}"
    )


print(
    "PASS: LBXTC attached XPT = "
    "7,387 valid / 745 missing / range 78-460."
)


print(
    "NOTE: CDC HTML currently reports range 44-460."
)

print(
    "      Fidelity remains anchored to the attached XPT."
)


# ============================================================
# 9. LBDTCSI VALID / MISSING CHECK
# ============================================================

#
# CDC HTML:
#
# Valid   = 7,387
# Missing = 745
# Range   = 1.14-11.9
#
#
# ATTACHED XPT:
#
# Valid   = 7,387
# Missing = 745
# Range   = 2.02-11.9
#
#
# Again: preserve the XPT.
#

s = df["LBDTCSI"]


si_valid = int(
    s.notna().sum()
)


si_missing = int(
    s.isna().sum()
)


si_min = s.min()
si_max = s.max()


print(
    "\n--- LBDTCSI CHECK ---"
)

print(
    "Valid:",
    f"{si_valid:,}"
)

print(
    "Missing:",
    f"{si_missing:,}"
)

print(
    "Attached XPT minimum:",
    si_min
)

print(
    "Attached XPT maximum:",
    si_max
)


if si_valid != 7387:

    raise ValueError(
        "LBDTCSI valid count differs."
    )


if si_missing != 745:

    raise ValueError(
        "LBDTCSI missing count differs."
    )


if not np.isclose(
    si_min,
    2.02,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected attached-XPT "
        f"LBDTCSI minimum: {si_min}"
    )


if not np.isclose(
    si_max,
    11.9,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected LBDTCSI maximum: {si_max}"
    )


print(
    "PASS: LBDTCSI attached XPT = "
    "7,387 valid / 745 missing / range 2.02-11.9."
)


print(
    "NOTE: CDC HTML currently reports range 1.14-11.9."
)

print(
    "      Fidelity remains anchored to the attached XPT."
)


# ============================================================
# 10. LBXTC WHOLE-NUMBER MEASUREMENT CHECK
# ============================================================

#
# All LBXTC results happen to be mathematically
# whole numbers.
#
# But this is a LAB MEASUREMENT.
#
# Therefore:
#
# 180.0 must remain 180.0
#
# NOT:
#
# 180
#

values = (
    df["LBXTC"]
    .dropna()
    .to_numpy()
)


fractional_count = int(
    (
        ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ).sum()
)


if fractional_count != 0:

    raise ValueError(
        "Unexpected fractional LBXTC value."
    )


print(
    "\nPASS: All 7,387 LBXTC values "
    "are mathematically whole-number measurements."
)

print(
    "      They will remain float-style values."
)


# ============================================================
# 11. LBDTCSI FRACTIONAL VALUE CHECK
# ============================================================

values = (
    df["LBDTCSI"]
    .dropna()
    .to_numpy()
)


fractional_mask = (
    ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )
)


si_fractional = int(
    fractional_mask.sum()
)


si_whole = int(
    (~fractional_mask).sum()
)


print(
    "\nLBDTCSI genuine fractional values:",
    f"{si_fractional:,}"
)

print(
    "LBDTCSI mathematically whole values:",
    f"{si_whole:,}"
)


if si_fractional != 7337:

    raise ValueError(
        "Unexpected LBDTCSI fractional count."
    )


if si_whole != 50:

    raise ValueError(
        "Unexpected LBDTCSI whole-number count."
    )


print(
    "PASS: LBDTCSI decimal pattern "
    "matches attached XPT."
)


# ============================================================
# 12. LEGITIMATE SOURCE .0 CHECK
# ============================================================

#
# Because BOTH variables are measurements:
#
# LBXTC:
# all 7,387 valid values should remain x.0
#
# LBDTCSI:
# 50 source values should remain x.0
#

EXPECTED_SOURCE_DOT_ZERO = {
    "LBXTC": 7387,
    "LBDTCSI": 50
}


print(
    "\n--- SOURCE MEASUREMENT .0 CHECK ---"
)


for col, expected_count in (
    EXPECTED_SOURCE_DOT_ZERO.items()
):

    actual_count = sum(
        str(float(value)).endswith(".0")
        for value in df[col].dropna()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected source .0 count.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate lab .0 counts identified."
)


# ============================================================
# 13. OPTIONAL DERIVED-FIELD CONSISTENCY INFORMATION
# ============================================================

#
# CDC documents:
#
# LBDTCSI = LBXTC * 0.02586
#
# But DO NOT recalculate or overwrite LBDTCSI.
#
# This section is informational only.
#

derived_calc = (
    df["LBXTC"]
    *
    0.02586
)


derived_difference = (
    df["LBDTCSI"]
    -
    derived_calc
).abs()


print(
    "\nMaximum difference from unrounded "
    "LBXTC * 0.02586:"
)

print(
    derived_difference.max()
)


print(
    "INFO: This check is informational only."
)

print(
    "      Released LBDTCSI values remain "
    "the source of truth."
)


# ============================================================
# 14. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# TCHOL_E requires NO numeric corrections.
#
# Tiny artifacts = 0
# Zero restorations = 0
#
# Therefore expected_df is EXACTLY the
# original attached XPT.
#

expected_df = source_df.copy()


# ============================================================
# 15. CLEAN SEQN ONLY
# ============================================================

#
# SEQN:
#
# 41475.0 -> 41475
#
#
# DO NOT alter:
#
# LBXTC
# LBDTCSI
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to clean integer formatting."
)

print(
    "PASS: LBXTC preserved as source-decoded float."
)

print(
    "PASS: LBDTCSI preserved as source-decoded float."
)


# ============================================================
# 16. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    expected_df["SEQN"].to_numpy(),
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN numeric values changed."
    )


for col in MEASUREMENT_COLUMNS:

    if not np.allclose(
        expected_df[col].to_numpy(),
        df[col].to_numpy(),
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: No underlying numeric value changed."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# NO:
#
# round()
# float_format
# decimal quantization
# recalculation
#
# Missing XPT numeric values become blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 18. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 3 columns numerically match "
    "the original attached XPT exactly."
)


# ============================================================
# 21. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 8132:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 23. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# SEQN:
#
# str(int(source_value))
#
#
# Laboratory measurements:
#
# str(float(source_value))
#
#
# Examples:
#
# 41475.0 -> "41475"
#
# LBXTC:
# 180.0 -> "180.0"
#
# LBDTCSI:
# 4.66 -> "4.66"
#
# 3.0 -> "3.0"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "TCHOL_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT representation."
)


# ============================================================
# 24. UNWANTED SEQN .0 CHECK
# ============================================================

seqn_dot_zero_count = sum(
    row["SEQN"].endswith(".0")
    for row in raw_rows
    if row["SEQN"] != ""
)


print(
    "\nUnwanted SEQN '.0' values:",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains .0 formatting."
    )


print(
    "PASS: No unwanted .0 in SEQN."
)


# ============================================================
# 25. LEGITIMATE LAB .0 CHECK
# ============================================================

print(
    "\n--- LEGITIMATE LAB .0 CHECK ---"
)


for col, expected_count in (
    EXPECTED_SOURCE_DOT_ZERO.items()
):

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate .0 formatting changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate lab .0 values preserved."
)


# ============================================================
# 26. STRICT LAB-MEASUREMENT RAW CHECK
# ============================================================

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = row[col]


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Total-cholesterol measurement "
        "representation changed."
    )


print(
    "PASS: Every cholesterol measurement "
    "is preserved AS IS."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 29. FINAL SOURCE RANGE CHECK AFTER EXPORT
# ============================================================

final_lbx = csv_df["LBXTC"]
final_si = csv_df["LBDTCSI"]


if not (
    int(final_lbx.notna().sum()) == 7387
    and
    int(final_lbx.isna().sum()) == 745
    and
    final_lbx.min() == 78
    and
    final_lbx.max() == 460
):

    raise ValueError(
        "Final LBXTC source values/counts changed."
    )


if not (
    int(final_si.notna().sum()) == 7387
    and
    int(final_si.isna().sum()) == 745
    and
    np.isclose(
        final_si.min(),
        2.02,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        final_si.max(),
        11.9,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "Final LBDTCSI source values/counts changed."
    )


print(
    "\nPASS: Final values/counts still "
    "match attached TCHOL_E XPT."
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TCHOL_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)


print(
    "\nLBXTC:"
)

print(
    "Valid = 7,387"
)

print(
    "Missing = 745"
)

print(
    "Attached XPT range = 78-460"
)

print(
    "CDC HTML range = 44-460"
)

print(
    "Legitimate .0 values = 7,387"
)


print(
    "\nLBDTCSI:"
)

print(
    "Valid = 7,387"
)

print(
    "Missing = 745"
)

print(
    "Attached XPT range = 2.02-11.9"
)

print(
    "CDC HTML range = 1.14-11.9"
)

print(
    "Genuine fractional values = 7,337"
)

print(
    "Legitimate .0 values = 50"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted SEQN .0:",
    seqn_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "TCHOL_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 TCHOL_E XPT -> CSV
Rows: 8,132
Columns: 3
PASS: TCHOL_E structure = 8,132 rows x 3 columns.

Identifier fields: ['SEQN']
Lab measurements preserved AS IS: ['LBXTC', 'LBDTCSI']

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: TCHOL_E contains NO tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original TCHOL_E contains no numeric zeros.

--- SEQN CHECK ---
PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- LBXTC CHECK ---
Valid: 7,387
Missing: 745
Attached XPT minimum: 78.0
Attached XPT maximum: 460.0
PASS: LBXTC attached XPT = 7,387 valid / 745 missing / range 78-460.
NOTE: CDC HTML currently reports range 44-460.
      Fidelity remains anchored to the attached XPT.

--- LBDTCSI CHECK ---
Valid: 7,387
Missing: 745
Attached XPT minimum: 2.02
Attached XPT maximum: 11.9
PASS: LBDTCSI attached XPT = 7,387 valid / 745 missing / range 2.02-11.9.
NOTE: CDC HTML currently reports range 1.14-11.9.
      Fidelity rem